# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

In [20]:
#download Forest Fire Dataset from UCI Machine Learning Repository
#extract data files into the subdirectory '05_src/data'
import os
import requests
import zipfile
import io

def download_forest_fire_dataset():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv'
    response = requests.get(url)
    response.raise_for_status()
    #create folders if they don't exist
    os.makedirs("../../05_src/data/fires", exist_ok=True)
    #write data to file
    with open("../../05_src/data/fires/forestfires.csv", 'wb') as f:
        f.write(response.content)

In [ ]:
pip install requests

In [ ]:
download_forest_fire_dataset()

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#load pipeline and other necessary libraries
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [33]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler

In [40]:
from sklearn.preprocessing import PolynomialFeatures

In [43]:
from sklearn.linear_model import LinearRegression

In [46]:
from sklearn.ensemble import RandomForestRegressor


In [49]:
from sklearn.model_selection import GridSearchCV


In [50]:
from sklearn.linear_model import Lasso


In [53]:
from sklearn.linear_model import LassoCV


In [ ]:
from sklearn.metrics import mean_squared_error

In [75]:
from sklearn.compose import TransformedTargetRegressor


In [26]:
pip install scikit-learn


  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 50.9 MB/s eta 0:00:00m eta 0:00:01
Using cached joblib-1.4.2-py3-none-any.whl (301 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 62.8 MB/s eta 0:00:000:00:01m eta 0:00:01
Using cached threadpoolctl-3.5.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [30]:
# create features and target variable
X = fires_dt.drop('area', axis = 1)
y = fires_dt['area']

# Split the data into training and test sets
# avoid data leakage by splitting the data before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [138]:
# transform the target variable usng log transformation
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [32]:
# Detect any missing values in each column of X_train
# Report the number of missing values in each column

missing_values = X_train.isnull().sum()
missing_values

# No need to impute missing values

coord_x    0
coord_y    0
month      0
day        0
ffmc       0
dmc        0
dc         0
isi        0
temp       0
rh         0
wind       0
rain       0
dtype: int64

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [131]:
# define numberic and categorical features
numeric_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']

# create transformers for numeric and categorical features
# For numeric variable: Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 

# use Robust Scaler for numeric features
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])


# For categorical variable: Apply one hot encoding to the categorical variables.
# create summer and winter months
# reference to https://www.kaggle.com/code/muhammadawon/forest-fires-predictions
summer = ['jun', 'jul', 'aug']

# apply summer months to the month column
X_train['summer'] = X_train['month'].apply(lambda x: 1 if x in summer else 0)
X_test['summer'] = X_test['month'].apply(lambda x: 1 if x in summer else 0)

categorical_features.append('summer')

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preproc1 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [60]:
preproc1

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [132]:
# create preproc2 for non-linear transformation
# use PolynomialFeatures to create non-linear transformation

preproc2_numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False))
])

preproc2 = ColumnTransformer(
    transformers=[
        ('num', preproc2_numeric_pipeline, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

preproc2




ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler()),
                                                 ('poly',
                                                  PolynomialFeatures(include_bias=False))]),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['month', 'day', 'summer'])])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [133]:
# Pipeline A = preproc1 + baseline

# add a step labelled preproccessing that applies preproc1 to the features
# add a step labelled model that applies a lasso linear regression model to the transformed features
# import lasso linear regression model


pipeline_A = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('model', Lasso())
])


In [82]:
pipeline_A

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['month', 'day'])])),
                ('regressor',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc 'expm1'>,
                                            regressor=Lasso(max_iter=10000)))])

In [134]:
# Pipeline B = preproc2 + baseline

pipeline_B = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('model', Lasso())
])

In [135]:
# Pipeline C = preproc1 + advanced model

# use RandomForestRegressor as the advanced model


pipeline_C = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('model', RandomForestRegressor())
])
    

In [130]:
# Pipeline D = preproc2 + advanced model

pipeline_D = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('model', RandomForestRegressor())
])


    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [141]:
# perform grid search to find the best model for each pipeline

# tune pipeline_A 

# Define the hyperparameter grid to tune Lasso's alpha
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# Set up GridSearchCV using 5-fold cross-validation and negative mean squared error as the scoring metric
grid_search = GridSearchCV(pipeline_A, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the grid search on the training data
grid_search.fit(X_train, y_train_log)

# Output the best parameters and the corresponding cross-validation score
print("Best parameters:", grid_search.best_params_)
print("Best CV score (negative MSE):", grid_search.best_score_)

pipeline_A_alpha = grid_search.best_params_["model__alpha"]



Best parameters: {'model__alpha': 1.0}
Best CV score (negative MSE): -1.8932420695838288


In [142]:
# tune pipeline_B


# Define the hyperparameter grid to tune Lasso's alpha
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# Set up GridSearchCV using 5-fold cross-validation and negative mean squared error as the scoring metric
grid_search = GridSearchCV(pipeline_B, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the grid search on the training data
grid_search.fit(X_train, y_train_log)

# Output the best parameters and the corresponding cross-validation score
print("Best parameters:", grid_search.best_params_)
print("Best CV score (negative MSE):", grid_search.best_score_)

pipeline_B_alpha = grid_search.best_params_["model__alpha"]


/Users/chukkalok/python/02_activities/assignments/myenv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.906e+01, tolerance: 6.170e-02
  model = cd_fast.enet_coordinate_descent(
/Users/chukkalok/python/02_activities/assignments/myenv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.189e+00, tolerance: 6.539e-02
  model = cd_fast.enet_coordinate_descent(
/Users/chukkalok/python/02_activities/assignments/myenv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the

Best parameters: {'model__alpha': 10.0}
Best CV score (negative MSE): -1.8932420695838288


In [144]:
# tune pipeline_C

# use GridSearchCV to tune RandomForestRegressor

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, 30, 40, 50]
}

grid_search = GridSearchCV(pipeline_C, param_grid, cv=5)
grid_search.fit(X_train, y_train_log)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

pipeline_C_params = grid_search.best_params_

Best parameters: {'model__max_depth': 10, 'model__n_estimators': 100}
Best cross-validation score: -0.12292238106573725


In [ ]:
# tune pipeline_D

grid_search = GridSearchCV(pipeline_D, param_grid, cv=5)
grid_search.fit(X_train, y_train_log)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

pipeline_D_params = grid_search.best_params_

# Evaluate

+ Which model has the best performance?

In [58]:
# Evaluate the best model for each pipeline on the test set
# use mean squared error as the evaluation metric

from sklearn.metrics import mean_squared_error

# pipeline_A, use the best alpha found in the grid search
pipeline_A = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('model', Lasso(alpha=pipeline_A_alpha))
])

pipeline_A.fit(X_train, y_train)
y_pred = pipeline_A.predict(X_test)

print("Pipeline A MSE:", mean_squared_error(y_test, y_pred))

# pipeline_B, use the best alpha found in the grid search
pipeline_B = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('model', Lasso(alpha=pipeline_B_alpha))
])

pipeline_B.fit(X_train, y_train)
y_pred = pipeline_B.predict(X_test)

print("Pipeline B MSE:", mean_squared_error(y_test, y_pred))

# pipeline_C, use the best parameters found in the grid search
pipeline_C = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('model', RandomForestRegressor(n_estimators= pipeline_C_params["model__n_estimators"], max_depth=pipeline_C_params["model__max_depth"]))
])

pipeline_C.fit(X_train, y_train)
y_pred = pipeline_C.predict(X_test)

print("Pipeline C MSE:", mean_squared_error(y_test, y_pred))

# pipeline_D, use the best parameters found in the grid search

pipeline_D = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('model', RandomForestRegressor(n_estimators=pipeline_D_params["model__n_estimators"], max_depth=pipeline_D_params["model__max_depth"]))
])

pipeline_D.fit(X_train, y_train)
y_pred = pipeline_D.predict(X_test)

print("Pipeline D MSE:", mean_squared_error(y_test, y_pred))


Pipeline A MSE: 11795.34511053629
Pipeline B MSE: 11934.703236084095
Pipeline C MSE: 11797.27798824383
Pipeline D MSE: 14242.985883210145


In [112]:
# pipeline_A, use the best alpha found in the grid search
pipeline_A = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', TransformedTargetRegressor(
        regressor=Lasso(alpha=pipeline_A_alpha, max_iter=10000),
        func=np.log1p,         # Apply log(1+x) transformation to the target during fitting
        inverse_func=np.expm1   # Use exp(x)-1 to transform predictions back to the original scale
    ))
])

pipeline_A.fit(X_train, y_train)
y_pred = pipeline_A.predict(X_test)

print("Pipeline A MSE:", mean_squared_error(y_test, y_pred))

Pipeline A MSE: 12049.177958940929


In [113]:
# pipeline_B, use the best alpha found in the grid search
pipeline_B = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('regressor', TransformedTargetRegressor(
        regressor=Lasso(alpha=pipeline_B_alpha, max_iter=10000),
        func=np.log1p,         # Apply log(1+x) transformation to the target during fitting
        inverse_func=np.expm1   # Use exp(x)-1 to transform predictions back to the original scale
    ))
])

pipeline_B.fit(X_train, y_train)
y_pred = pipeline_B.predict(X_test)

print("Pipeline B MSE:", mean_squared_error(y_test, y_pred))

Pipeline B MSE: 12096.849877793442


In [96]:
# pipeline_C, use the best parameters found in the grid search
pipeline_C = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('model', RandomForestRegressor(n_estimators= pipeline_C_params["regressor__regressor__n_estimators"], max_depth=pipeline_C_params["regressor__regressor__max_depth"]))
])

pipeline_C.fit(X_train, y_train)
y_pred = pipeline_C.predict(X_test)

print("Pipeline C MSE:", mean_squared_error(y_test, y_pred))


Pipeline C MSE: 11742.654027286915


In [111]:
# pipeline_D, use the best parameters found in the grid search

pipeline_D = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('model', RandomForestRegressor(n_estimators=pipeline_D_params["regressor__regressor__n_estimators"], max_depth=pipeline_D_params["regressor__regressor__max_depth"]))
])

pipeline_D.fit(X_train, y_train)
y_pred = pipeline_D.predict(X_test)

print("Pipeline D MSE:", mean_squared_error(y_test, y_pred))


Pipeline D MSE: 14053.807282080475


# Export

+ Save the best performing model to a pickle file.

In [114]:
# Save the best performing model to a pickle file.

import joblib

# Save the best performing model to a pickle file
joblib.dump(pipeline_C, "pipeline_C.pkl")



['pipeline_C.pkl']

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [116]:
pip install shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 57.9 MB/s eta 0:00:00m eta 0:00:010:00:01
  Created wheel for shap: filename=shap-0.46.0-cp313-cp313-macosx_13_0_arm64.whl size=457912 sha256=96697898634d26b89e1bd4ea05b752573c22012fa3f2d6ec82eeb8874366fa92
  Stored in directory: /Users/chukkalok/Library/Caches/pip/wheels/c7/ee/d7/e89bc04111b0f4f786aa49028712ea4abb638428dcbeb0d1a8
Successfully built shap

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [120]:
# Use SHAP values to explain the following only for the best-performing model:

# Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

# Use the SHAP library to explain the model's predictions on the test set.

import shap

# load the best model
model = joblib.load("pipeline_C.pkl")

# explain the model's predictions on the test set
explainer = shap.Explainer(model.named_steps['model'])
shap_values = explainer(X_test)




KeyError: 'model'

In [122]:
model.predict

<bound method Pipeline.predict of Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['month', 'day'])])),
                ('regressor',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc

In [126]:
# Ensure all numeric columns are properly converted to numeric types
numeric_columns = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
X_test[numeric_columns] = X_test[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Ensure categorical columns are properly converted to category types
categorical_columns = ['month', 'day']
X_test[categorical_columns] = X_test[categorical_columns].astype('category')

background = X_test.sample(n=100, random_state=42)

explainer = shap.Explainer(model.predict, background)

shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test)

TypeError: unsupported operand type(s) for -: 'str' and 'str'

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.